In [53]:
import pandas as pd
from datasets import Dataset
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation data.csv')


In [54]:
df

,user_input,retrieved_contexts,response,reference
0,"Who is the current CEO of Setopia Co., Ltd.?",NaN,NaN,"The current CEO of Setopia Co., Ltd. is Sang-c..."
1,"What was the company name of Setopia Co., Ltd....",NaN,NaN,"It was Midas AI Co., Ltd."
2,"When was the merger date for Setopia Co., Ltd....",NaN,NaN,"The merger date was January 2, 2023."
3,What is the main product of Setopia's steel bu...,NaN,NaN,"The main product is STS 201, and it holds the ..."
4,The distribution business Setopia entered into...,NaN,NaN,It was an exclusive domestic distributorship c...
5,Who was the auditor for the 11th fiscal period...,NaN,NaN,"The auditor was Samil PwC, and the audit opini..."
6,What happened to the contract for the acquisit...,NaN,NaN,"The contract was terminated on January 20, 2025."
7,What was the ratio of the capital reduction wi...,NaN,NaN,A 5-to-1 capital reduction was completed.
8,"Who is the largest shareholder of Setopia Co.,...",NaN,NaN,"The largest shareholder is SA Corporation Co.,..."
9,"How much was the fine imposed on Setopia Co., ...",NaN,NaN,A fine of 270 million KRW was imposed related ...


In [52]:
import sys
import os
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
from pipeline.document_loader import DocumentLoader
from pipeline.chunker import Chunker
from pipeline.embedder import Embedder
from pipeline.vector_retriever import VectorRetriever
from pipeline.generator import Generator
import config
from tkinter import filedialog

def setup_pipeline():
    chunker = Chunker(
        chunk_size = config.CHUNK_SIZE,
        overlap_size = config.OVERLAP_SIZE
    )
    print('Chunking Complete')
    embedder = Embedder(
        model_name = config.EMBEDDING_MODEL
    )
    print('Embedding Complete')
    retriever =VectorRetriever(
        db_path = config.DB_PATH,
        model_name = config.EMBEDDING_MODEL,
        collection_name = config.COLLECTION_NAME
    )
    print('Retrieving Coplete')
    generator = Generator(
        model_name = config.LLM_NAME,
        options = config.DEFAULT_OLLAMA_OPTIONS
    )
    return chunker, embedder, retriever, generator

def run_indexing(file_path, chunker, embedder, retriever):
    loader = DocumentLoader(file_path = file_path)
    document = loader.load()
    chunks = chunker.chunking(document)
    embedded_chunks = embedder.embed_documents(chunks)
    file_name = os.path.basename(file_path)
    retriever.add_documents(embedded_chunks, file_name)

chunker, embedder, retriever, generator = setup_pipeline()

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
run_indexing(file_path, chunker, embedder, retriever)

Chunking Complete
Embedding Complete
Retrieving Coplete


In [55]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query)
    outputs = generator.generate(retrieved_data, query)
    outputs = json.loads(outputs)
    if 'used_context' not in outputs.keys():
        outputs['used_context'] = []
    elif 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')




/tmp/ipykernel_981/4201832838.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 5I. 회사의 개요 1. 회사의 개요 가. 연결대상 종속회사 개황 가-1. 연결대상 종속회사 현황(요약) 주식회사 세토피아 (이하 ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
/tmp/ipykernel_981/4201832838.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'response'] = outputs['answer']


progress : 1/30
progress : 2/30
progress : 3/30
progress : 4/30
progress : 5/30
progress : 6/30
progress : 7/30
progress : 8/30
progress : 9/30
progress : 10/30
progress : 11/30
progress : 12/30
progress : 13/30
progress : 14/30
progress : 15/30
progress : 16/30
progress : 17/30
progress : 18/30
progress : 19/30
progress : 20/30
progress : 21/30
progress : 22/30
progress : 23/30
progress : 24/30
progress : 25/30
progress : 26/30
progress : 27/30
progress : 28/30
progress : 29/30
progress : 30/30


In [57]:
dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    else (json.loads(x) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

# reference: 항상 문자열로
if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )